# Notebook 12 — Latency / Throughput Pareto Frontiers

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 11 classified streaming distribution regimes before policy selection.

Notebook 12 asks a runtime-systems question:

> when latency and throughput conflict, which execution policies remain Pareto-efficient?

Constraint view:
> adaptive execution should not optimize one metric blindly; it should expose tradeoff frontiers.

## Goals

1. Load Notebook 11 online classification outputs when available.
2. Generate policy candidates per window/regime:
   - scalar
   - SIMD
   - coherent-local
   - guarded fallback
   - hybrid
3. Estimate latency, throughput, pressure, and coherence per candidate.
4. Compute Pareto-efficient policies:
   - maximize throughput
   - minimize latency
   - minimize pressure
   - preserve coherence
5. Compare policy choices under explicit objective weights.
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 11 online classification table

If missing, this notebook creates a fallback window table.

In [ ]:
online_path = RESULTS_DIR / "notebook11_online_distribution_classification.csv"

if online_path.exists():
    windows = pd.read_csv(online_path)
    print("Loaded:", online_path)
else:
    print("Notebook 11 output not found; using fallback windows.")
    regimes = ["low_entropy_repeating", "sequential_ids", "uniform_32bit", "zipfian_smallints", "clustered_ranges"]
    rows = []
    for i in range(100):
        reg = regimes[i % len(regimes)]
        rows.append({
            "window_id": i,
            "truth_regime": reg,
            "predicted_regime": reg,
            "coherence_score": {
                "low_entropy_repeating": 0.95,
                "sequential_ids": 0.55,
                "uniform_32bit": 0.08,
                "zipfian_smallints": 0.42,
                "clustered_ranges": 0.18,
            }[reg],
            "hardware_pressure_proxy": {
                "low_entropy_repeating": 0.02,
                "sequential_ids": 0.30,
                "uniform_32bit": 0.88,
                "zipfian_smallints": 0.72,
                "clustered_ranges": 0.98,
            }[reg],
            "regime_confidence": 0.95,
        })
    windows = pd.DataFrame(rows)

windows.head()

## Generate candidate policies

Every window receives several possible execution candidates.
The candidate table lets us compare tradeoffs instead of choosing a policy immediately.

In [ ]:
policies = ["scalar", "simd", "coherent_local", "guarded_fallback", "hybrid"]

base = {
    "low_entropy_repeating": {
        "scalar": (1650, 0.62, 0.12),
        "simd": (1400, 0.75, 0.28),
        "coherent_local": (1750, 0.58, 0.08),
        "guarded_fallback": (1200, 0.92, 0.18),
        "hybrid": (1500, 0.70, 0.20),
    },
    "sequential_ids": {
        "scalar": (1350, 0.74, 0.24),
        "simd": (1450, 0.69, 0.30),
        "coherent_local": (1300, 0.78, 0.20),
        "guarded_fallback": (1100, 0.95, 0.34),
        "hybrid": (1400, 0.71, 0.25),
    },
    "uniform_32bit": {
        "scalar": (1100, 0.95, 0.62),
        "simd": (1900, 0.53, 0.42),
        "coherent_local": (1000, 1.02, 0.70),
        "guarded_fallback": (1200, 0.90, 0.50),
        "hybrid": (1650, 0.62, 0.48),
    },
    "zipfian_smallints": {
        "scalar": (1200, 0.82, 0.58),
        "simd": (1500, 0.67, 0.52),
        "coherent_local": (1250, 0.80, 0.45),
        "guarded_fallback": (1150, 0.88, 0.48),
        "hybrid": (1550, 0.64, 0.44),
    },
    "clustered_ranges": {
        "scalar": (900, 1.12, 0.90),
        "simd": (950, 1.05, 0.92),
        "coherent_local": (850, 1.20, 0.88),
        "guarded_fallback": (1200, 0.86, 0.62),
        "hybrid": (1050, 0.98, 0.75),
    },
}

rng = np.random.default_rng(42)
rows = []

for _, row in windows.iterrows():
    reg = row.get("predicted_regime", row.get("truth_regime", "unknown"))
    if reg not in base:
        reg = row.get("truth_regime", "sequential_ids")
    coherence = float(row.get("coherence_score", 0.5))
    pressure = float(row.get("hardware_pressure_proxy", 0.5))
    confidence = float(row.get("regime_confidence", 0.9))

    for pol in policies:
        tp, lat, pol_pressure = base[reg][pol]
        # Small deterministic-ish jitter to avoid totally degenerate frontiers.
        jitter = rng.normal(0, 0.015)
        adj_throughput = tp * (1.0 + jitter)
        adj_latency = max(0.01, lat * (1.0 - jitter))
        adj_pressure = np.clip(0.6 * pol_pressure + 0.4 * pressure, 0, 1)
        adj_coherence = np.clip(0.65 * coherence + 0.35 * (1.0 - adj_pressure), 0, 1)

        rows.append({
            "window_id": int(row["window_id"]),
            "truth_regime": row.get("truth_regime", reg),
            "predicted_regime": reg,
            "policy": pol,
            "throughput": adj_throughput,
            "latency": adj_latency,
            "pressure": adj_pressure,
            "coherence": adj_coherence,
            "regime_confidence": confidence,
        })

candidates = pd.DataFrame(rows)
candidates.head()

## Pareto frontier computation

A candidate is Pareto-efficient if no other candidate in the same window has:

- greater or equal throughput,
- less or equal latency,
- less or equal pressure,
- greater or equal coherence,

with at least one strict improvement.

In [ ]:
def pareto_mask(group):
    vals = group[["throughput", "latency", "pressure", "coherence"]].to_numpy(float)
    n = len(vals)
    efficient = np.ones(n, dtype=bool)
    for i in range(n):
        if not efficient[i]:
            continue
        # Objectives: throughput high, latency low, pressure low, coherence high.
        dominates_i = (
            (vals[:, 0] >= vals[i, 0]) &
            (vals[:, 1] <= vals[i, 1]) &
            (vals[:, 2] <= vals[i, 2]) &
            (vals[:, 3] >= vals[i, 3]) &
            (
                (vals[:, 0] > vals[i, 0]) |
                (vals[:, 1] < vals[i, 1]) |
                (vals[:, 2] < vals[i, 2]) |
                (vals[:, 3] > vals[i, 3])
            )
        )
        dominates_i[i] = False
        if dominates_i.any():
            efficient[i] = False
    return pd.Series(efficient, index=group.index)

candidates["pareto_efficient"] = (
    candidates
    .groupby("window_id", group_keys=False)
    .apply(pareto_mask)
    .astype(bool)
)

candidates[["window_id", "policy", "pareto_efficient"]].head(10)

## Weighted objective policy selection

We compare three explicit objective profiles:

- **throughput_first**
- **latency_first**
- **balanced_constraint**

In [ ]:
def norm_by_window(df, col, invert=False):
    out = []
    for _, g in df.groupby("window_id"):
        s = g[col].astype(float)
        lo, hi = s.min(), s.max()
        if hi == lo:
            vals = np.ones(len(s))
        else:
            vals = (s - lo) / (hi - lo)
        if invert:
            vals = 1.0 - vals
        out.extend(vals)
    return np.array(out)

candidates["throughput_score"] = norm_by_window(candidates, "throughput", invert=False)
candidates["latency_score"] = norm_by_window(candidates, "latency", invert=True)
candidates["pressure_score"] = norm_by_window(candidates, "pressure", invert=True)
candidates["coherence_score_obj"] = norm_by_window(candidates, "coherence", invert=False)

objective_profiles = {
    "throughput_first": {
        "throughput_score": 0.60,
        "latency_score": 0.15,
        "pressure_score": 0.10,
        "coherence_score_obj": 0.15,
    },
    "latency_first": {
        "throughput_score": 0.20,
        "latency_score": 0.55,
        "pressure_score": 0.15,
        "coherence_score_obj": 0.10,
    },
    "balanced_constraint": {
        "throughput_score": 0.30,
        "latency_score": 0.25,
        "pressure_score": 0.20,
        "coherence_score_obj": 0.25,
    },
}

for profile, weights in objective_profiles.items():
    candidates[f"objective_{profile}"] = sum(candidates[k] * w for k, w in weights.items())

selections = []
for profile in objective_profiles:
    col = f"objective_{profile}"
    idx = candidates.groupby("window_id")[col].idxmax()
    sel = candidates.loc[idx].copy()
    sel["objective_profile"] = profile
    sel["objective_score"] = sel[col]
    selections.append(sel)

selected = pd.concat(selections, ignore_index=True)
selected.head()

## Export Pareto candidate and selection tables

In [ ]:
candidates_csv_path = RESULTS_DIR / "notebook12_pareto_candidates.csv"
candidates_json_path = RESULTS_DIR / "notebook12_pareto_candidates.json"
selected_csv_path = RESULTS_DIR / "notebook12_pareto_selected_policies.csv"

candidates.to_csv(candidates_csv_path, index=False)
candidates.to_json(candidates_json_path, orient="records", indent=2)
selected.to_csv(selected_csv_path, index=False)

print("Saved:", candidates_csv_path)
print("Saved:", candidates_json_path)
print("Saved:", selected_csv_path)

## Figure 1 — Latency / throughput candidate cloud

Pareto-efficient points are shown with larger markers.

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook12_latency_throughput_cloud.png"

plt.figure(figsize=(9, 6))
non = candidates[~candidates["pareto_efficient"]]
par = candidates[candidates["pareto_efficient"]]
plt.scatter(non["latency"], non["throughput"], s=20, alpha=0.35, label="dominated")
plt.scatter(par["latency"], par["throughput"], s=70, alpha=0.85, label="pareto-efficient")
plt.xlabel("Latency (lower is better)")
plt.ylabel("Throughput (higher is better)")
plt.title("Latency / Throughput Policy Candidate Cloud")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Pareto policy frequency

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook12_pareto_policy_frequency.png"

freq = (
    candidates[candidates["pareto_efficient"]]
    .groupby("policy", as_index=False)
    .agg(count=("policy", "size"))
    .sort_values("count")
)

plt.figure(figsize=(8, 5))
plt.bar(freq["policy"], freq["count"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Pareto-efficient count")
plt.title("Pareto-Efficient Policy Frequency")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Objective-profile policy choices

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook12_objective_policy_choices.png"

choice = pd.crosstab(selected["objective_profile"], selected["policy"])

plt.figure(figsize=(9, 5))
bottom = np.zeros(len(choice.index))
x = np.arange(len(choice.index))
for pol in choice.columns:
    plt.bar(x, choice[pol].values, bottom=bottom, label=pol)
    bottom += choice[pol].values
plt.xticks(x, choice.index, rotation=20, ha="right")
plt.ylabel("Selected windows")
plt.title("Policy Choices by Objective Profile")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Pareto frontier by regime

Average selected objective scores by predicted regime.

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook12_frontier_score_by_regime.png"

score_by_regime = (
    selected
    .groupby(["predicted_regime", "objective_profile"], as_index=False)
    .agg(mean_objective=("objective_score", "mean"))
)

pivot = score_by_regime.pivot(index="predicted_regime", columns="objective_profile", values="mean_objective").fillna(0)

plt.figure(figsize=(9, 5))
plt.imshow(pivot.values, aspect="auto", vmin=0, vmax=1)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha="right")
plt.colorbar(label="Mean objective score")
plt.title("Pareto Objective Scores by Regime")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Tradeoff timeline

Compare selected policies over time for each objective profile.

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook12_policy_timeline_by_objective.png"

policy_labels = sorted(selected["policy"].unique())
policy_to_id = {p: i for i, p in enumerate(policy_labels)}

plt.figure(figsize=(12, 5))
for profile in selected["objective_profile"].unique():
    part = selected[selected["objective_profile"] == profile].sort_values("window_id")
    plt.step(part["window_id"], part["policy"].map(policy_to_id), where="mid", label=profile)

plt.yticks(list(policy_to_id.values()), list(policy_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Selected policy")
plt.title("Policy Timeline by Objective Profile")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_12_latency_throughput_pareto_frontiers.md"

summary = {
    "candidate_rows": int(len(candidates)),
    "windows": int(candidates["window_id"].nunique()),
    "pareto_efficient_rows": int(candidates["pareto_efficient"].sum()),
    "pareto_fraction": float(candidates["pareto_efficient"].mean()),
    "objective_profiles": int(len(objective_profiles)),
}

pareto_freq = candidates[candidates["pareto_efficient"]]["policy"].value_counts().rename_axis("policy").reset_index(name="pareto_count")
selected_freq = pd.crosstab(selected["objective_profile"], selected["policy"])

lines = [
    "# Report 12 — Latency / Throughput Pareto Frontiers",
    "",
    "This report analyzes policy tradeoffs between throughput, latency, pressure, and coherence.",
    "",
    "Constraint view:",
    "> adaptive execution should not optimize one metric blindly; it should expose tradeoff frontiers.",
    "",
    "## Generated outputs",
    "",
    f"- Candidate CSV: `{candidates_csv_path}`",
    f"- Candidate JSON: `{candidates_json_path}`",
    f"- Selected policy CSV: `{selected_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Pareto-efficient policy frequency",
    "",
    pareto_freq.to_markdown(index=False),
    "",
    "## Selected policies by objective profile",
    "",
    selected_freq.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Pareto frontiers expose which policies remain useful under multi-objective constraints.",
    "- Throughput-first, latency-first, and balanced profiles can select different execution paths for the same window.",
    "- Guarded fallback can be Pareto-efficient when pressure dominates fragmented regimes.",
    "- SIMD is not universally optimal; it is optimal under objective profiles and regimes that reward wide throughput.",
    "",
    "## Next step",
    "",
    "Notebook 13 can introduce mixed-regime decomposition: classify windows that contain blends rather than single regimes.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook12_latency_throughput_pareto_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook12_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_12_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))